In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

pd.set_option("display.max_columns", 100)
pd.set_option('display.float_format', lambda x: f"{x:.3f}")

In [ ]:
FILEPATH= "../data/raw/Fuel_consumption.xlsx"

df1 = pd.read_excel(FILEPATH, sheet_name='Sheet1')
df2 = pd.read_excel(FILEPATH, sheet_name='Sheet2')

In [ ]:
df1

In [ ]:
df2

In [ ]:
df = pd.merge(left=df2, right=df1, on='Snr')
df.shape

In [ ]:
df.drop(labels='Snr', axis=1, inplace=True)

In [ ]:
df.Year.unique()

In [ ]:
df.drop(labels='Year', axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df[df.duplicated()]

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
df[df.duplicated()]

In [ ]:
df.head()

In [ ]:
df.info()

## Separating Features

In [ ]:
df.dtypes

In [ ]:
cont_cols = []
cat_cols = []
for i in df.columns:
    if (df[i].dtypes=='float64') or (df[i].dtypes=='int64'):
        cont_cols.append(i)
    else:
        cat_cols.append(i)

In [ ]:
print(cont_cols)
print(cat_cols)

## Categorical columns analysis

In [ ]:
df[cat_cols].describe().T

In [ ]:
plt.figure(figsize=(15, 6))
plt.suptitle('Univariate Analysis of Categorical Features', fontsize=20, fontweight='bold', alpha=0.8, y=1)

for i,col in enumerate(cat_cols):
    plt.subplot(2, 2, i+1)
    sns.countplot(x=col, data=df, hue=col, palette="Set2", legend=False)
    plt.xlabel(col)
    plt.xticks(rotation=90)
    plt.tight_layout()

os.makedirs("reports", exist_ok=True)
plt.savefig("reports/categorical_counts.png", dpi=300, bbox_inches="tight")

### Conclusion: Make & Model features are showing high cardinality

### checking for any numerical entries into categorical columns

In [ ]:
for col in cat_cols:
    print("_____{}______".format(col))
    for i in df[col]:
        if (type(i)==int) or (type(i)==float):
            print(i)

In [ ]:
df[df['MODEL']==626]

In [ ]:
# Dropping rows
df = df[df['MODEL']!=626]
df = df.reset_index(drop=True)

In [ ]:
for col in cat_cols:
    print("_____{}______".format(col))
    for i in df[col]:
        if i.isnumeric():
            print(i)

### Grouping models based on companies

In [ ]:
pd.DataFrame(df.groupby(by=['MAKE','MODEL'])['COEMISSIONS'].aggregate('mean')).sort_values(by='COEMISSIONS')

In [ ]:
fig = px.sunburst(df, 
                  path=["MAKE", "MODEL"],
                  values="COEMISSIONS",
                  color="MAKE")
fig.update_layout( width=900, height=900,
                 title={ "text": "MODELS by Companies", 
                         "x": 0.5, # center 
                         "y": 0.95, # vertical position 
                         "xanchor": "center", 
                         "yanchor": "top", 
                         "font": dict(size=24, color="darkblue", family="Arial")
                       }
                 )

os.makedirs("reports", exist_ok=True)
fig.write_image("reports/models.png")

fig.show()

### Analyzing effect of models on coemission

In [ ]:
# Frequency analysis
df['MODEL'].value_counts(normalize=True)*100

In [ ]:
# Top 20 high frequency models
df['MODEL'].value_counts().head(20)

In [ ]:
import squarify

freq = df['MODEL'].value_counts().head(20)

colors = [
    '#4E79A7', '#F28E2B', '#E15759', '#76B7B2',
    '#59A14F', '#EDC948', '#B07AA1', '#FF9DA7'
]

squarify.plot(sizes=freq.values, label=freq.index, color=colors[:len(freq)])
plt.axis('off')
os.makedirs("reports", exist_ok=True)
plt.savefig("reports/top_20_frequent_models.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
freq = df['MODEL'].value_counts()

plt.figure(figsize=(15, 10))
plt.suptitle('Frequency plot of models', fontsize=20, fontweight='bold', alpha=0.8, y=1)

plt.subplot(2,1,1)
plt.plot(freq.values)
plt.yscale('log')   # optional but useful
plt.title('Model Frequency Distribution')
plt.xlabel('Category Rank')
plt.ylabel('Frequency')

plt.subplot(2,1,2)
plt.bar(x=freq.index, height=freq.values)
plt.xlabel('Categories')
plt.ylabel('Count')

os.makedirs("reports", exist_ok=True)
plt.savefig("reports/model_frequency_distribution.png", dpi=300, bbox_inches="tight")

plt.show()

### The frequency distrbution of models are right skewed, i.e. few categories have high frequency whereas most of the categories have low frequency

### Now checking contrbituion of models in coemmision sorted accordin to respective frequency

In [ ]:
model_freq_df = pd.DataFrame(df.groupby(by=['MODEL'])['COEMISSIONS'].aggregate('mean'))
model_freq_df = model_freq_df.reset_index()
model_freq_df

In [ ]:
labels = list(df.MODEL.value_counts().index)
values = list(df.MODEL.value_counts().values)
freq_dict = {}
for label, value in zip(labels, values):
    freq_dict[label] = int(value)
model_freq_df['MODEL_freq'] = model_freq_df['MODEL'].map(freq_dict)
model_freq_df

In [ ]:
plt.figure(figsize=(15, 6))
plt.suptitle('Model Frequency vs Coemission', fontsize=20, fontweight='bold', alpha=0.8, y=1)

sns.barplot(x=model_freq_df.MODEL_freq, y=model_freq_df.COEMISSIONS, hue=model_freq_df.MODEL_freq, legend=False)
os.makedirs("reports", exist_ok=True)
plt.savefig("reports/model_frequency_vs_coemission.png", dpi=300, bbox_inches="tight")
plt.show()

### Conclusion: Models should be encoded according to its contribution in the coemssion

### Analyzing "Make" column

In [ ]:
freq = df['MAKE'].value_counts()

plt.figure(figsize=(15, 10))
plt.suptitle('Frequency plot of make', fontsize=20, fontweight='bold', alpha=0.8, y=1)

plt.subplot(2,1,1)
plt.plot(freq.values)
plt.yscale('log')   # optional but useful
plt.title('Make Frequency Distribution')
plt.xlabel('Category Rank')
plt.ylabel('Frequency')

plt.subplot(2,1,2)
plt.bar(x=freq.index, height=freq.values)
plt.xlabel('Categories')
plt.ylabel('Count')

os.makedirs("reports", exist_ok=True)
plt.savefig("reports/make_frequency_distribution.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
make_freq_df = pd.DataFrame(df.groupby(by=['MAKE'])['COEMISSIONS'].aggregate('mean'))
make_freq_df = make_freq_df.reset_index()
make_freq_df

In [ ]:
labels = list(df.MAKE.value_counts().index)
values = list(df.MAKE.value_counts().values)
freq_dict = {}
for label, value in zip(labels, values):
    freq_dict[label] = int(value)
make_freq_df['MAKE_freq'] = make_freq_df['MAKE'].map(freq_dict)
make_freq_df

In [ ]:
plt.figure(figsize=(15, 6))
plt.suptitle('Make Frequency vs Coemission', fontsize=20, fontweight='bold', alpha=0.8, y=1)

sns.barplot(x=make_freq_df.MAKE_freq, y=make_freq_df.COEMISSIONS, hue=make_freq_df.MAKE_freq, legend=False)
os.makedirs("reports", exist_ok=True)
plt.savefig("reports/make_frequency_vs_coemission.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(20,6))
sns.violinplot(x='MAKE', y='COEMISSIONS', data=df, inner='box')
plt.xticks(rotation=90)
plt.title("CO2 Emission Distribution by Make")
os.makedirs("reports", exist_ok=True)
plt.savefig("reports/CO2_Emission_Distribution_by_Make.png", dpi=300, bbox_inches="tight")
plt.show()

### Conclusion: Make should be encoded according to its contribution in the coemssion

## Analysis of transmission

In [ ]:
plt.figure(figsize=(15,10))
plt.subplot(2,1,1)
sns.barplot(x="TRANSMISSION", y="COEMISSIONS", data=df, estimator=np.mean, palette='Set2', hue="TRANSMISSION")
plt.subplot(2,1,2)
sns.violinplot(x='TRANSMISSION', y='COEMISSIONS', data=df, inner='box')
plt.title("Mean Coemssion Vs Transmission")

os.makedirs("reports", exist_ok=True)
plt.savefig("reports/Mean_Coemssion_Vs_Transmission.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
pd.DataFrame(df.groupby(by=['TRANSMISSION'])['COEMISSIONS'].aggregate('mean')).sort_values(by='COEMISSIONS')

### Conclusion: We should perform ordinal encoding

### Analyzing fuel column

In [ ]:
plt.figure(figsize=(20,10))
plt.subplot(2,1,1)
sns.barplot(x="FUEL", y="COEMISSIONS", data=df, estimator=np.mean, palette='Set2', hue="FUEL")
plt.subplot(2,1,2)
sns.violinplot(x='FUEL', y='COEMISSIONS', data=df, inner='box')
plt.xticks(rotation=90)
plt.title("CO2 Emission Distribution by Fuel")
os.makedirs("reports", exist_ok=True)
plt.savefig("reports/CO2_Emission_Distribution_by_Fuel.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
pd.DataFrame(df.groupby(by=['FUEL'])['COEMISSIONS'].aggregate('mean')).sort_values(by='COEMISSIONS')

### Conclusion: perform Ordinal encoding

### Analyzing Fuel vs Transmission vs Coemission

In [ ]:
plt.figure(figsize=(20,6))
sns.barplot(x="TRANSMISSION", y="COEMISSIONS", data=df, estimator=np.mean, palette='Set2', hue="FUEL")
os.makedirs("reports", exist_ok=True)
plt.savefig("reports/TRANSMISSION_vs_FUEL_vs_COEMISSION.png", dpi=300, bbox_inches="tight")
plt.show()

### Analyzing numerical columns

In [ ]:
plt.figure(figsize=(15, 6))
plt.suptitle('Univariate Analysis of Numerical Features', fontsize=20, fontweight='bold', alpha=0.8, y=1)

for i,col in enumerate(cont_cols):
    plt.subplot(2, 2, i+1)
    sns.kdeplot(data=df[col],fill=True, color='r')
    plt.xlabel(col)
    plt.xticks(rotation=90)
    plt.tight_layout()

os.makedirs("reports", exist_ok=True)
plt.savefig("reports/numerical_column_distribution.png", dpi=300, bbox_inches="tight")

In [ ]:
sns.pairplot(df[cont_cols])

os.makedirs("reports", exist_ok=True)
plt.savefig("reports/paiplot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
cols = ['ENGINE SIZE', 'CYLINDERS', 'FUEL CONSUMPTION']
os.makedirs("reports", exist_ok=True)
for i,col in enumerate(cols, start=1):
    sns.jointplot(x=df[col], y=df['COEMISSIONS'], kind='reg', line_kws={"color": "red"})
    plt.savefig(f"reports/{col}.png", dpi=300, bbox_inches="tight")

In [ ]:
df[cont_cols].corr()['COEMISSIONS']

In [ ]:
sns.boxplot(data=df[cont_cols])
plt.show()

In [ ]:
value = pd.qcut(df['FUEL CONSUMPTION'], q=4)
value

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x=value, y=df['COEMISSIONS'])
plt.xticks(rotation=45)
plt.show()

In [ ]:
EXPORT_FILEPATH="../data/cleaned/"
df.to_excel(EXPORT_FILEPATH+"df_cleaned.xlsx", index=False)